# Hindsight on Oracle 23ai

This cookbook runs **Hindsight** — a memory layer for AI agents — on **Oracle Database 23ai**.
All of that memory lives in Oracle, using native database features: vector embeddings in a
`VECTOR` column with an HNSW index, keyword search through **Oracle Text**, and entity
relationships as a graph across ordinary tables — so there's no separate vector database to run.

We'll use a **customer-support** example: an agent that stores and recalls what it knows about a
customer, *Acme Corp*. We showcase each Hindsight feature — `retain`, `recall`, `reflect`,
mental models — how each changes which memories come back, and the raw Oracle SQL underneath.
The finale: Hindsight **reasoning across** those memories to surface something we never wrote down.


## Prerequisites

This notebook talks to a **Hindsight server configured to use Oracle Database 23ai** as its
backend. You point Hindsight at Oracle with two environment variables when you start the server:

```
HINDSIGHT_API_DATABASE_BACKEND=oracle
HINDSIGHT_API_DATABASE_URL=oracle+oracledb://<user>:<password>@<host>:1521/<service>
```

Migrations run automatically on startup, emitting Oracle-native DDL (a `VECTOR` column, an HNSW
index, an Oracle Text index). Then set `HINDSIGHT_API_URL` below to point at that server (default
`http://localhost:8888`), and install the client:


In [ ]:
!pip install hindsight-client python-oracledb nest_asyncio python-dotenv sentence-transformers -q

We also open a **read-only Oracle connection** alongside the Hindsight client, so we can
look at the raw tables and see exactly what Hindsight stored.


In [ ]:
import nest_asyncio; nest_asyncio.apply()
import os, array, textwrap
from dotenv import load_dotenv; load_dotenv()
from hindsight_client import Hindsight

HINDSIGHT_API_URL = os.getenv("HINDSIGHT_API_URL", "http://localhost:8888")
client = Hindsight(base_url=HINDSIGHT_API_URL)
BANK = "acme-support"

# Read-only Oracle connection, used ONLY for the "look, it's in Oracle" peeks.
import oracledb
oracledb.defaults.fetch_lobs = False
ORA = dict(user=os.getenv("ORACLE_USER","hindsight_test"),
           password=os.getenv("ORACLE_PASS","hindsight_test"),
           dsn=os.getenv("ORACLE_DSN","localhost:1521/FREEPDB1"))

def sql(query, binds=None, title=None, width=66, max_rows=25):
    conn = oracledb.connect(**ORA); cur = conn.cursor()
    try:
        cur.execute(query, binds or {}); cols=[d[0] for d in cur.description]; rows=cur.fetchmany(max_rows)
    finally:
        cur.close(); conn.close()
    if title: print(title + "\n" + "-"*len(title))
    print("  ".join(c[:width] for c in cols))
    for r in rows: print("  ".join(("" if v is None else str(v))[:width] for v in r))
    print(f"\n({len(rows)} row(s))"); return rows

def show(res, limit=8):
    """Print recall results, de-duplicated, so each distinct memory shows once."""
    seen=set()
    for r in res.results:
        k=(r.text or "").strip().lower()[:70]
        if k in seen: continue
        seen.add(k); print(f"  · [{r.type}] {r.text}")
        if len(seen)>=limit: break

print("Connected to", HINDSIGHT_API_URL, "| Oracle peek as", ORA["user"])

## 1 · Teach the agent — `retain`

`retain` is how the agent *learns*. You hand it text in plain English, and rather than storing
it verbatim, Hindsight uses an LLM to pull it apart: it extracts the **atomic facts**, the
**entities** and relationships involved, and **when** things happened, computes a **vector
embedding** for each, and writes all of that into Oracle. So one sentence in can become several
structured, searchable memories.

We teach it some context about Acme **plus three separate support incidents** from over the
months. Read the incidents (#4, #5, #6) — we'll come back to them at the end.


In [ ]:
from datetime import datetime, timezone
def d(y, m, day): return datetime(y, m, day, tzinfo=timezone.utc)

# (content, context, tags, when). Note: the three incidents are logged independently — nothing
# here says they're connected.
memories = [
    ("Acme Corp runs Oracle Database 23ai on Oracle Cloud Infrastructure (OCI) in the Ashburn region.",
     "infrastructure setup", ["infrastructure"], d(2025, 9, 15)),                                          # #1 INFRA
    ("Acme's lead DBA is Priya Nair; she prefers to be contacted by email, not phone.",
     "onboarding call", ["contact"], d(2025, 9, 15)),                                                      # #2 CONTACT
    ("Acme must remain HIPAA-compliant: all customer PII has to be encrypted at rest.",
     "security review", ["security", "compliance"], d(2025, 11, 3)),                                       # #3 SECURITY
    ("On Jan 20 2026, Acme's nightly batch job hit repeated ORA-00060 deadlocks on the ORDERS table.",
     "incident #4821", ["incident"], d(2026, 1, 20)),                                                      # #4 INCIDENT
    ("On Feb 15 2026, Acme's overnight inventory sync timed out waiting on row locks on the ORDERS table.",
     "incident #4960", ["incident"], d(2026, 2, 15)),                                                      # #5 INCIDENT
    ("On Mar 12 2026, a scheduled 2am analytics query on the ORDERS table caused lock contention and was killed.",
     "incident #5077", ["incident"], d(2026, 3, 12)),                                                      # #6 INCIDENT
    ("Acme plans to launch a product-recommendation feature built on Oracle 23ai vector search.",
     "roadmap sync", ["roadmap"], d(2026, 5, 28)),                                                         # #7 ROADMAP
]
for i, (content, context, tags, when) in enumerate(memories, 1):
    client.retain(bank_id=BANK, content=content, context=context, tags=tags, timestamp=when)
    print(f"#{i:<2} {when:%Y-%m-%d}  ({', '.join(tags)})  <-  {content[:52]}...")
print(f"\nThe agent now has 7 memories about Acme, stored in Oracle.")

### What landed in Oracle

The memories are now rows in `MEMORY_UNITS`, each with a native Oracle `VECTOR` embedding;
the entities Hindsight pulled out (Acme, ORDERS, Priya Nair, …) are in `ENTITIES`. Note the
count — we taught it **7** things, but there are *more* than 7 memory units, because `retain`
decomposed each input into its atomic facts.


In [ ]:
sql("SELECT (SELECT COUNT(*) FROM memory_units WHERE bank_id=:b) AS memory_units, "
    "       (SELECT COUNT(*) FROM entities WHERE bank_id=:b) AS entities "
    "FROM dual", binds={"b": BANK}, title="Stored in Oracle for this bank")
print()
sql("SELECT VECTOR_DIMENSION_COUNT(embedding) AS vec_dims, VECTOR_DIMENSION_FORMAT(embedding) AS fmt "
    "FROM memory_units WHERE bank_id=:b FETCH FIRST 1 ROWS ONLY", binds={"b": BANK},
    title="EMBEDDING is a native Oracle 23ai VECTOR")

## 2 · How `recall` finds the right memories

Storing memories is the easy part — the interesting part is *retrieval*. When the agent asks a
question, `recall` doesn't just string-match: it combines several retrieval techniques —
**semantic** vector search, **keyword** search, the **entity graph**, and **time** — and fuses
the results into one ranked list.

Let's break those techniques down one at a time. Each is backed by Oracle, so for each we'll run
the actual SQL underneath — and you'll see how each technique changes *which* memories come back.


### 2a · Semantic search — *matches meaning, not keywords*

We ask about **"performance problems."** **None** of the memories contain the word "performance,"
so a keyword search finds nothing — but Hindsight embedded each memory as an Oracle `VECTOR`, so
it matches on *meaning* and ranks the **incidents** first. Here it is as pure Oracle SQL: embed
the question, then `ORDER BY VECTOR_DISTANCE`.


In [ ]:
from sentence_transformers import SentenceTransformer
_st = SentenceTransformer(os.getenv("HINDSIGHT_API_EMBEDDINGS_LOCAL_MODEL", "BAAI/bge-small-en-v1.5"))
qvec = _st.encode("are there any database performance problems?")     # same model the server uses
qbind = array.array("f", [float(x) for x in qvec])

sql("SELECT ROUND(VECTOR_DISTANCE(embedding, :qv, COSINE), 3) AS distance, SUBSTR(text,1,58) AS memory "
    "FROM memory_units WHERE bank_id=:b ORDER BY distance FETCH FIRST 4 ROWS ONLY",
    binds={"qv": qbind, "b": BANK},
    title="Closest memories to 'performance problems' (native Oracle VECTOR_DISTANCE)")
# -> the incidents (#4/#5/#6) rank first even though none says "performance" — semantic search.

### 2b · Keyword search — *exact terms, when precision matters*

Sometimes you want the literal term. Hindsight also maintains an **Oracle Text** index, so you can
match exact words with `CONTAINS()`. Searching **"HIPAA"** returns the compliance memory (#3) — and
*only* it.


In [ ]:
sql("SELECT SCORE(1) AS relevance, SUBSTR(text,1,60) AS memory "
    "FROM memory_units WHERE bank_id=:b AND CONTAINS(text, 'HIPAA', 1) > 0 "
    "ORDER BY relevance DESC", binds={"b": BANK},
    title="Oracle Text CONTAINS('HIPAA') — exact keyword match")
# -> only #3. Semantic + keyword together = recall finds the right memory whether you remember
#    the exact words or just the gist.

### 2c · Tags — *scope memory to a topic*

When we taught the agent, we tagged each memory. The same broad question returns totally
different memories depending on the tag filter:


In [ ]:
print(">>> tags=['security']:")
show(client.recall(bank_id=BANK, query="what should I know about Acme?", budget="mid", tags=["security"]))
print("\n>>> tags=['roadmap']:")
show(client.recall(bank_id=BANK, query="what should I know about Acme?", budget="mid", tags=["roadmap"]))
# -> 'security' -> #3 (HIPAA);  'roadmap' -> #7 (the vector-search launch).

### 2d · The knowledge graph — *connections between things*

While learning, Hindsight built a **graph**: the entities it found and which memories mention
them — ordinary Oracle tables. Notice **ORDERS** already stands out (it shows up in all three
incidents) — a hint of what's coming.


In [ ]:
sql("SELECT e.canonical_name AS entity, COUNT(ue.unit_id) AS in_memories "
    "FROM entities e LEFT JOIN unit_entities ue ON ue.entity_id = e.id "
    "WHERE e.bank_id=:b GROUP BY e.canonical_name ORDER BY in_memories DESC, e.canonical_name "
    "FETCH FIRST 8 ROWS ONLY", binds={"b": BANK},
    title="Entities Hindsight extracted, and how many memories mention each")

### 2e · Time — *Hindsight knows **when** things happened*

We stamped each memory with an event date. So "what's the **latest**?" favors recent memories,
and the whole timeline is right there in Oracle.


In [ ]:
print(">>> 'What is the latest with Acme?' (recency-aware recall):")
show(client.recall(bank_id=BANK, query="what is the latest development with Acme?", budget="mid"))
print()
sql("SELECT TO_CHAR(event_date,'YYYY-MM-DD') AS happened, SUBSTR(text,1,52) AS memory "
    "FROM memory_units WHERE bank_id=:b ORDER BY event_date DESC FETCH FIRST 7 ROWS ONLY",
    binds={"b": BANK}, title="Acme's memory timeline (event_date in Oracle)")

## 3 · `recall` — get the relevant facts for a question

In real use you don't pick a lens by hand: `recall` runs **all** of them at once — semantic,
keyword, graph, temporal — then does the part the raw SQL above *doesn't* show: it fuses the
separate result lists with **Reciprocal Rank Fusion** and re-orders them with a **cross-encoder
reranker**.

Ask it a **specific** question and it returns just the memories that fit — not everything we
stored. Here we ask only about Acme's database problems, and that's what comes back (no contact,
compliance, or roadmap notes):


In [ ]:
recall = client.recall(
    bank_id=BANK, query="What recurring database problems has Acme run into?",
    budget="mid", max_tokens=200,
)
print(f"Recalled {len(recall.results)} facts (deduped view):")
show(recall)

## 4 · `reflect` — reasoning that goes **beyond** what we stored  ⭐

This is the part that makes Hindsight more than a database. `recall` returns memories;
**`reflect` reasons across them.**

We logged three incidents weeks apart, as separate notes — **nowhere did we tell Hindsight they
were related.** So we'll just ask a normal account-management question and see what it does:


In [ ]:
# A plain, non-leading question — no mention of patterns or the ORDERS table.
reco = client.reflect(
    bank_id=BANK, budget="mid",
    query=("Based on the issues we've seen with Acme, what proactive recommendations would you "
           "make? Give me a bullet list — for each point, state the issue you found and your "
           "recommendation."),
)
print(reco.text)

Notice what happened: to answer, Hindsight connected the three separate incidents and
surfaced the common thread on its own — **they're all lock contention on the `ORDERS` table
during nightly/overnight jobs** — then recommended fixes for it. That root cause appears in
*none* of the individual memories; it came from reasoning across them.


## 5 · Mental models — *a durable summary the agent maintains*

A **mental model** is a living summary Hindsight keeps for you. You name it and give it a
`source_query`; the server synthesizes the content from the memories and refreshes it as new
ones arrive. Like everything else, it lives in Oracle — with its own native `VECTOR`.


In [ ]:
import time
client.create_mental_model(
    bank_id=BANK, name="Acme Corp — Account Snapshot",
    source_query="everything important about Acme Corp: environment, contacts, recurring issues, security, plans",
)
print("Created; the server is synthesizing the snapshot from all the memories...")
model=None
for _ in range(40):
    items=client.list_mental_models(bank_id=BANK).items
    model=items[0] if items else None
    if model and model.content and not model.content.startswith("Generating"): break
    time.sleep(3)
if model:
    print(f"\n• {model.name}\n")
    print(textwrap.fill(model.content or "(still generating)", width=92, initial_indent="  ", subsequent_indent="  "))
print()
sql("SELECT name, VECTOR_DIMENSION_COUNT(embedding) AS vec_dims FROM mental_models WHERE bank_id=:b",
    binds={"b": BANK}, title="The mental model is an Oracle row, with its own VECTOR")

## 6 · Cleanup

In [ ]:
client.delete_bank(BANK)
print(f"Deleted bank '{BANK}'.")